# 01 — Corpus and chunking

This notebook explores the data that powers AskML RAG. It does **not** alter any generated files; it reads the canonical documents and chunks created by the ingestion and chunking scripts.

By the end, you should be able to explain the path from a public source to a retrievable chunk, why chunk metadata matters for citations, and how the current word-based chunking strategy works.

Start Jupyter from the repository root with `uv run jupyter lab`, then run the cells in order.

## 1. Locate the project and load generated data

The helper below makes the notebook work whether Jupyter starts in the repository root or in `notebooks/`. Canonical documents are one JSON file per source; chunks are stored as JSON Lines (one JSON object per line).

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Markdown, display

from askml_rag.models import CanonicalDocument, Chunk

In [2]:
def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find the AskML RAG project root.")

In [3]:
PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHUNKS_PATH = PROCESSED_DIR / "chunks" / "chunks.jsonl"
MANIFEST_PATH = PROCESSED_DIR / "chunks" / "chunking_manifest.json"

documents = [
    CanonicalDocument.model_validate_json(path.read_text(encoding="utf-8"))
    for path in sorted(PROCESSED_DIR.glob("*.json"))
]
chunks = [
    Chunk.model_validate_json(line)
    for line in CHUNKS_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Canonical documents: {len(documents)}")
print(f"Retrieval chunks: {len(chunks)}")

Project root: /home/mlares/store/Learning/AI_engineering/askml_rag
Canonical documents: 5
Retrieval chunks: 28


## 2. Inspect the canonical corpus

A canonical document is the normalized version of one source. It keeps the provenance needed to trace an answer back to its source: ID, title, URL, type, topics, visibility, and content hash. The body is normalized Markdown.

In [4]:
document_table = pd.DataFrame(
    {
        "document_id": document.document_id,
        "title": document.title,
        "type": document.document_type.value,
        "topics": ", ".join(document.topics),
        "characters": len(document.body_markdown),
        "hash_prefix": document.content_hash[:12],
    }
    for document in documents
)
document_table

,document_id,title,type,topics,characters,hash_prefix
0,skills,Marcelo Lares,website,"work-culture, skills, leadership, ownership",8151,d079c9020d37
1,website_home,Marcelo Lares,website,"applied-ai, technical-leadership, data-science...",6002,b35e03a1c545
2,website_projects,Selected projects,website,"applied-ai, production-machine-learning, techn...",22901,d4ed1016758f
3,website_research,Research,website,"astronomy, astrostatistics, machine-learning, ...",6545,2aa9a6c11c0b
4,website_teaching,Teaching and mentoring,website,"teaching, mentoring, data-science, statistics",3256,3d64c87c5018


Choose a document and inspect its beginning. Notice that this is a **normalized representation**, not necessarily a byte-for-byte copy of the raw source. The hash lets us detect when this normalized content changes.

In [8]:
document_by_id = {document.document_id: document for document in documents}
selected_document_id = documents[0].document_id
selected_document = document_by_id[selected_document_id]

display(Markdown(f"## {selected_document.title}"))
display(Markdown(selected_document.body_markdown[:2_000]))

## Marcelo Lares

# Marcelo Lares — Skills Review and Retrieval Keywords
## Profile summary
Marcelo Lares is a \*\*Lead Data Scientist / Applied AI / Machine Learning Engineering\*\* professional with a PhD in Astronomy and experience across scientific research, university teaching, technical leadership, consulting, and production machine learning. His strongest differentiator is combining \*\*statistical rigor, experimentation, reproducibility, and business-oriented ML delivery\*\*.
His experience spans end-to-end work: problem framing, data analysis, feature engineering, model development, offline evaluation, A/B testing, deployment support, rollout, monitoring, documentation, and communication with technical and non-technical stakeholders.
## Core strengths
\* Data Science
\* Machine Learning
\* Applied AI
\* Production Machine Learning
\* Machine Learning Engineering
\* Statistical Learning
\* Statistical Inference
\* Bayesian Inference
\* Experimental Design
\* Statistical Experimentation
\* A/B Testing
\* Model Evaluation
\* Offline Evaluation
\* Online Evaluation
\* Uncertainty Assessment
\* Reproducibility
\* Experiment Tracking
\* Data-Driven Decision Making
\* Feature Engineering
\* End-to-End Model Validation
\* Model Deployment
\* Production Rollout
\* Model Monitoring
\* Technical Leadership
\* Cross-Functional Collaboration
\* Mentoring
\* University Teaching
\* Data Storytelling
\* Data Visualization
\* Technical Communication
## Machine learning and modeling
Demonstrated experience with:
\* Supervised Learning
\* Classification
\* Regression
\* Time Series Forecasting
\* Ranking Models
\* Learning to Rank
\* Recommendation Systems
\* Recommender Systems
\* Retrieval and Ranking
\* Embeddings
\* Feature Stores
\* Ensemble Methods
\* Tree-Based Models
\* Random Forest
\* XGBoost
\* Support Vector Machines
\* SVM
\* Logistic Regression
\* Naive Bayes
\* Linear Discriminant Analysis
\* LDA
\* Quadratic Discriminant Analysis
\* QDA
\* Principal Component Analysis
\* PCA
\*

## 3. Understand chunking with a small example

The current baseline splits on whitespace. A chunk contains at most `chunk_size_words`; the final words of one chunk are repeated in the next chunk as overlap. Overlap helps preserve a thought that crosses a boundary.

This is intentionally simple. Later, we can compare it with section-aware or token-based chunking, but a transparent baseline gives us something measurable first.

In [9]:
from askml_rag.retrieval.chunking import split_into_word_chunks

toy_text = "one two three four five six seven eight nine ten eleven twelve"
toy_chunks = split_into_word_chunks(
    toy_text,
    chunk_size_words=5,
    overlap_words=2,
)

pd.DataFrame(
    {
        "chunk_index": range(len(toy_chunks)),
        "text": toy_chunks,
        "word_count": [len(chunk.split()) for chunk in toy_chunks],
    }
)

,chunk_index,text,word_count
0,0,one two three four five,5
1,1,four five six seven eight,5
2,2,seven eight nine ten eleven,5
3,3,ten eleven twelve,3


## 4. Summarize the real chunk corpus

Chunks carry their parent document metadata. This is why a retriever can return text for the model **and** enough information to create a source citation later.

In [10]:
chunk_table = pd.DataFrame(
    {
        "chunk_id": chunk.chunk_id,
        "document_id": chunk.document_id,
        "chunk_index": chunk.chunk_index,
        "title": chunk.title,
        "topics": ", ".join(chunk.topics),
        "words": len(chunk.text.split()),
    }
    for chunk in chunks
)

chunk_summary = (
    chunk_table.groupby("document_id", as_index=False)
    .agg(
        chunks=("chunk_id", "count"),
        mean_words=("words", "mean"),
        max_words=("words", "max"),
    )
    .sort_values("document_id")
)
chunk_summary

,document_id,chunks,mean_words,max_words
0,skills,6,223.000000,250
1,website_home,4,212.000000,250
2,website_projects,13,243.692308,250
3,website_research,3,221.000000,250
4,website_teaching,2,201.000000,250


In [12]:
def show_chunk(chunk_id: str) -> None:
    chunk = next(chunk for chunk in chunks if chunk.chunk_id == chunk_id)
    display(Markdown(f"### `{chunk.chunk_id}`"))
    display(
        Markdown(
            f"**Document:** `{chunk.document_id}`  \n"
            f"**Source:** {chunk.source_url or 'no public URL recorded'}  \n"
            f"**Topics:** {', '.join(chunk.topics) or 'none'}"
        )
    )
    display(Markdown(chunk.text))

In [15]:
show_chunk(chunks[0].chunk_id)

### `skills_chunk_000`

**Document:** `skills`  
**Source:** no public URL recorded  
**Topics:** work-culture, skills, leadership, ownership

# Marcelo Lares — Skills Review and Retrieval Keywords ## Profile summary Marcelo Lares is a \*\*Lead Data Scientist / Applied AI / Machine Learning Engineering\*\* professional with a PhD in Astronomy and experience across scientific research, university teaching, technical leadership, consulting, and production machine learning. His strongest differentiator is combining \*\*statistical rigor, experimentation, reproducibility, and business-oriented ML delivery\*\*. His experience spans end-to-end work: problem framing, data analysis, feature engineering, model development, offline evaluation, A/B testing, deployment support, rollout, monitoring, documentation, and communication with technical and non-technical stakeholders. ## Core strengths \* Data Science \* Machine Learning \* Applied AI \* Production Machine Learning \* Machine Learning Engineering \* Statistical Learning \* Statistical Inference \* Bayesian Inference \* Experimental Design \* Statistical Experimentation \* A/B Testing \* Model Evaluation \* Offline Evaluation \* Online Evaluation \* Uncertainty Assessment \* Reproducibility \* Experiment Tracking \* Data-Driven Decision Making \* Feature Engineering \* End-to-End Model Validation \* Model Deployment \* Production Rollout \* Model Monitoring \* Technical Leadership \* Cross-Functional Collaboration \* Mentoring \* University Teaching \* Data Storytelling \* Data Visualization \* Technical Communication ## Machine learning and modeling Demonstrated experience with: \* Supervised Learning \* Classification \* Regression \* Time Series Forecasting \* Ranking Models \* Learning to Rank \* Recommendation Systems \* Recommender Systems \* Retrieval and Ranking \* Embeddings \* Feature Stores \* Ensemble Methods \* Tree-Based Models \* Random Forest \* XGBoost \* Support Vector Machines \* SVM \* Logistic Regression \* Naive Bayes \* Linear Discriminant

## 5. Check reproducibility metadata

The chunking manifest records the configuration and source-document hashes used to create the current chunk corpus. If a canonical document changes, re-run chunking; the manifest helps identify why chunk IDs or results may have changed.

In [16]:
chunking_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
chunking_manifest

{'strategy': 'word_overlap',
 'chunk_size_words': 250,
 'overlap_words': 40,
 'document_count': 5,
 'chunk_count': 28,
 'documents': [{'document_id': 'skills',
   'content_hash': 'd079c9020d373f911a496e85d242058a41918ba0641082972b77f665d3e1db29'},
  {'document_id': 'website_home',
   'content_hash': 'b35e03a1c545b775d9fb1f7b256710fc50eb027112ebf3a011a1be0275f84eae'},
  {'document_id': 'website_projects',
   'content_hash': 'd4ed1016758f7a4ee1012cb0b09f6fac4eae21c686996d879e4633539ed8ad5d'},
  {'document_id': 'website_research',
   'content_hash': '2aa9a6c11c0b391b545feaffd4c2b8b63600779c3d6ceae81746ea5d9ed904ba'},
  {'document_id': 'website_teaching',
   'content_hash': '3d64c87c50181dcd9dba879ef9b714ec7cb7fc6a80172810a225d30581349258'}]}

## Questions to explore

1. Which document produces the most chunks, and why might that affect retrieval?
2. Inspect a chunk near the end of a long document. Does it remain understandable without its neighbouring chunks?
3. Change the toy overlap from 2 to 0 and then to 4. What information is preserved or duplicated?
4. If you changed the real chunk size, which benchmark labels would need to be reviewed?

Next: open `02_retrieval_comparison.ipynb` to search these chunks with BM25, embeddings, and hybrid retrieval.